<a href="https://colab.research.google.com/github/blkfin/courses/blob/main/cs675/exercises/deepagent-tutorial/deepagent_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Building AI Agents with DeepAgent
### CS 675 — Agentic AI | Hands-On Tutorial

---

In this tutorial, you'll go from **zero to a working AI agent** using LangChain's [DeepAgent](https://docs.langchain.com/oss/python/deepagents/overview) framework.

**What you'll build:**
1. A basic agent with custom tools
2. An agent that plans and tracks its own work
3. An agent that reads and writes files to manage context
4. An agent that delegates tasks to subagents
5. A research agent that combines all four capabilities

**Time:** ~50 minutes

**Prerequisites:** Python basics, understanding of what an agent is (tool-calling loop, planning, memory — everything from our lectures)

---
## Part 0: Setup

First, install DeepAgent and configure your API key.

In [ ]:
# Install the deepagents package
!pip install -qU deepagents tavily-python

In [ ]:
import os
from getpass import getpass

# ── Choose your model provider ──────────────────────────────────────
# Uncomment ONE of the following blocks.

# Option A: Anthropic (Claude)
# !pip install -qU "langchain[anthropic]"
# os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
# MODEL = "anthropic:claude-sonnet-4-6"  # default

# Option B: OpenAI
# !pip install -qU "langchain[openai]"
# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
# MODEL = "openai:gpt-4o-mini"

# Option C: Google Gemini (free tier available)
# !pip install -qU "langchain[google-genai]"
# os.environ["GOOGLE_API_KEY"] = getpass("Google API key: ")
# MODEL = "google-genai:gemini-2.0-flash"

 # Option D: OpenRouter
# !pip install -qU "langchain[openai]"
# os.environ["OPENAI_API_KEY"] = getpass("OpenRouter API key: ")
# os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
# MODEL = "openai:anthropic/claude-sonnet-4-6"  # or "openai:google/gemini-2.0-flash", "openai:openai/gpt-4o-mini"

# Option E: Groq
# !pip install -qU "langchain[openai]"
# os.environ["OPENAI_API_KEY"] = getpass("Groq API key: ")
# os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
MODEL = "openai:llama-3.3-70b-versatile"  # or "openai:mixtral-8x7b-32768"

print(f"Using model: {MODEL}")

In [ ]:
# Optional: Tavily for web search (used in later sections)
# Get a free key at https://tavily.com
os.environ["TAVILY_API_KEY"] = getpass("Tavily API key (or press Enter to skip): ") or "skip"
HAS_TAVILY = os.environ["TAVILY_API_KEY"] != "skip"
print("Tavily search:", "enabled ✓" if HAS_TAVILY else "disabled (some examples will use mock data)")

In [ ]:
# ── Helper: pretty-print agent responses ───────────────────────────
from IPython.display import Markdown, display


def show(result):
    """Display the agent's final response as formatted Markdown."""
    messages = result["messages"]
    # Walk backward to find the last AI message
    for msg in reversed(messages):
        if hasattr(msg, "content") and isinstance(msg.content, str) and msg.content.strip():
            display(Markdown(msg.content))
            return
    print("(no text response)")


def show_steps(result):
    """Show every message in the agent's trajectory — useful for seeing the loop."""
    for i, msg in enumerate(result["messages"]):
        role = type(msg).__name__
        content = getattr(msg, "content", "")
        if isinstance(content, list):  # tool-call blocks
            content = "\n".join(
                f"  → {c.get('type', '?')}: {str(c.get('text', c.get('name', '')))[:120]}"
                for c in content
            )
        elif isinstance(content, str):
            content = content[:200] + ("..." if len(content) > 200 else "")
        print(f"[{i}] {role}: {content}")
        # Show tool calls if present
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_preview = str(tc.get("args", {}))[:150]
                print(f"     🔧 {tc['name']}({args_preview})")

---
## Part 1: Your First Agent — The Tool-Calling Loop

Remember the **agent loop** from lecture?

```
User message → LLM thinks → calls a tool → gets result → LLM thinks again → responds
```

DeepAgent wraps this loop for you. All you provide is:
- **Tools** — Python functions the agent can call
- **A system prompt** — instructions for how the agent should behave

Let's start with the simplest possible agent: one tool, one question.

In [ ]:
from deepagents import create_deep_agent


# ── Define a tool ──────────────────────────────────────────────────
# Any Python function with a docstring becomes a tool.
# The docstring is what the LLM reads to decide when/how to use it.

def get_class_info(course_code: str) -> str:
    """Look up information about a university course by its code (e.g., 'CS675', 'IT612')."""
    catalog = {
        "CS675": {
            "title": "Data Science",
            "instructor": "Prof. Jones",
            "topics": ["Agent architectures", "Tool use", "Memory systems", "Multi-agent coordination"],
            "students": 16,
        },
        "IT612": {
            "title": "Scripting",
            "instructor": "Prof. Smith",
            "topics": ["Linux", "Bash scripting", "JavaScript", "Monitoring"],
            "students": 15,
        },
    }
    course = catalog.get(course_code.upper().replace(" ", ""))
    if course:
        return (
            f"{course_code}: {course['title']}\n"
            f"Instructor: {course['instructor']}\n"
            f"Topics: {', '.join(course['topics'])}\n"
            f"Enrolled: {course['students']} students"
        )
    return f"Course {course_code} not found in catalog."


# ── Create the agent ───────────────────────────────────────────────
agent = create_deep_agent(
    model=MODEL,
    tools=[get_class_info],
    system_prompt="You are a helpful university assistant. Use your tools to answer questions about courses.",
)

# ── Run it ─────────────────────────────────────────────────────────
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What topics does CS 675 cover?"}]}
)

show(result)

### 🔍 See the Loop in Action

The agent didn't just answer — it **called a tool** and used the result. Let's see every step:

In [ ]:
show_steps(result)

**What just happened:**
1. The user asked a question
2. The LLM decided to call `get_class_info("CS675")`
3. The function returned course data
4. The LLM used that data to compose a natural-language answer

That's the **agent loop**. Every agent you'll ever build — no matter how complex — is this loop, repeated.

---

### ✏️ Exercise 1: Add a Second Tool

Create a `get_professor_info` tool that returns information about a professor (office hours, research interests, etc.). Then create an agent with **both** tools and ask it a question that requires using both.

**Hint:** The agent will automatically decide which tools to call based on the question. Try: *"Who teaches CS 675 and what are their office hours?"*

In [ ]:
# YOUR CODE HERE

def get_professor_info(name: str) -> str:
    """Look up information about a professor by name."""
    # TODO: Create a dictionary of professors with office hours, research interests, etc.
    pass


# Create an agent with BOTH tools
# agent = create_deep_agent(...)

# Ask a question that requires both tools
# result = agent.invoke(...)

---
## Part 2: Planning — Agents That Organize Their Work

Real tasks aren't one question → one tool call. They're multi-step:

> *"Research three AI frameworks and write a comparison."*

DeepAgent includes a built-in **`write_todos`** tool. The agent uses it to:
- Break a task into steps
- Track which steps are done
- Adapt the plan as it learns new information

You don't need to enable this — it's on by default. Let's see it work.

In [ ]:
import json


# ── Some mock "research" tools ──────────────────────────────────────

def search_papers(query: str) -> str:
    """Search for academic papers on a topic. Returns titles, authors, and key findings."""
    # In a real agent, this would call an API like Semantic Scholar or arXiv
    papers_db = {
        "agent memory": [
            {"title": "MemGPT: Towards LLMs as Operating Systems", "year": 2023,
             "finding": "Virtual context management enables unbounded conversation history"},
            {"title": "Cognitive Architectures for Language Agents", "year": 2024,
             "finding": "Agents benefit from separating working memory, episodic memory, and procedural memory"},
            {"title": "Graph-Based Agent Memory (Yang et al.)", "year": 2026,
             "finding": "Knowledge graphs outperform flat retrieval for complex reasoning tasks"},
        ],
        "agent planning": [
            {"title": "Tree of Thoughts", "year": 2023,
             "finding": "Deliberate search over reasoning paths improves problem-solving"},
            {"title": "Plan-and-Solve Prompting", "year": 2023,
             "finding": "Explicit planning steps reduce errors in multi-step reasoning"},
            {"title": "SELF-RAG: Self-Reflective Retrieval-Augmented Generation", "year": 2023,
             "finding": "On-demand retrieval with self-reflection outperforms always-retrieve"},
        ],
        "multi-agent": [
            {"title": "AutoGen: Multi-Agent Conversation Framework", "year": 2023,
             "finding": "Conversational patterns between agents can solve complex tasks"},
            {"title": "CrewAI: Role-Based Multi-Agent Orchestration", "year": 2024,
             "finding": "Role specialization improves multi-agent task completion"},
            {"title": "Agent-as-a-Judge", "year": 2024,
             "finding": "Agents can evaluate other agents' outputs for quality control"},
        ],
    }
    # Fuzzy match: check if any key is a substring of the query
    for key, papers in papers_db.items():
        if key in query.lower():
            return json.dumps(papers, indent=2)
    return json.dumps([{"title": "No papers found", "finding": f"Try searching for: {list(papers_db.keys())}"}])


def summarize_topic(topic: str, notes: str) -> str:
    """Generate a one-paragraph summary of a topic given research notes."""
    return f"Summary of '{topic}': Based on the research notes provided, {notes[:200]}..."


# ── Create a research agent ────────────────────────────────────────
research_agent = create_deep_agent(
    model=MODEL,
    tools=[search_papers, summarize_topic],
    system_prompt=(
        "You are a research assistant. When given a complex task:\n"
        "1. Use write_todos to plan your steps FIRST\n"
        "2. Execute each step, updating todos as you go\n"
        "3. Synthesize your findings into a clear response\n"
        "Be thorough but concise."
    ),
)

# ── Give it a multi-step task ──────────────────────────────────────
result = research_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Research the three main areas of agent AI: memory systems, planning, "
        "and multi-agent coordination. Find key papers in each area, then write "
        "a brief comparison of the three areas."
    }]}
)

show(result)

In [ ]:
# See the full trajectory — notice the write_todos calls
show_steps(result)

**Key insight:** The agent created a plan *before* doing any work, then executed it step by step. This is the same pattern you see in production agents like Claude Code, Devin, and OpenClaw — **plan, then execute, then synthesize**.

The `write_todos` tool gives the agent a structured way to track progress. Without it, the agent would try to do everything in one shot — which works for simple tasks but falls apart on complex ones.

---

---
## Part 3: Filesystem — Agents That Manage Their Own Context

Here's a problem you've seen in lecture: **the context window is finite**.

If an agent researches 20 papers, it can't hold all the text in its conversation. But it can **write notes to files** and **read them back** when needed.

DeepAgent gives agents a **virtual filesystem** with these tools:
- `ls` — list files
- `read_file` — read file content
- `write_file` — create a new file
- `edit_file` — modify an existing file
- `glob` / `grep` — search for files or content

By default, files live **in memory** (ephemeral — gone when the session ends). But you can swap to local disk, a database, or a sandbox.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend


# ── Tools that produce large outputs ───────────────────────────────

def fetch_article(url: str) -> str:
    """Fetch the text content of a web article. Returns the full article text."""
    # Simulated articles — in production you'd use requests + BeautifulSoup
    articles = {
        "react-pattern": (
            "The ReAct Pattern: Reasoning + Acting in LLMs\n\n"
            "The ReAct pattern interleaves reasoning traces with tool-calling actions. "
            "At each step, the agent: (1) observes the current state, (2) reasons about "
            "what to do next, (3) takes an action (tool call), and (4) observes the result. "
            "This loop continues until the task is complete.\n\n"
            "Key advantages: interpretability (you can read the reasoning), grounding "
            "(actions provide real data), and error recovery (the agent can detect and "
            "correct mistakes). Key limitations: sequential execution, context window "
            "consumption, and sensitivity to tool descriptions.\n\n"
            "Production implementations include LangChain agents, Claude Code's tool loop, "
            "and OpenClaw's Pi agent runtime. Most modern agent frameworks use ReAct as "
            "their default execution pattern."
        ),
        "agent-memory": (
            "Memory Systems for AI Agents: A Survey\n\n"
            "Agent memory can be categorized into three tiers:\n"
            "1. Working Memory: The current conversation context. Fast but limited by "
            "the context window (typically 128K-1M tokens).\n"
            "2. Episodic Memory: Records of past interactions. Can be stored as conversation "
            "logs, summaries, or structured events. Retrieval is key — vector search, "
            "keyword search, or hybrid approaches.\n"
            "3. Semantic Memory: Distilled knowledge — concepts, relationships, procedures. "
            "Often stored as knowledge graphs or structured documents.\n\n"
            "The challenge is routing: given a query, which memory tier should the agent "
            "consult? Production systems use manifest-based routing (explicit pointers), "
            "semantic search (embedding similarity), or hybrid approaches."
        ),
    }
    for key, text in articles.items():
        if key in url.lower():
            return text
    return f"Could not fetch article from {url}. Try: react-pattern, agent-memory"


# ── Create agent with filesystem awareness ─────────────────────────
writer_agent = create_deep_agent(
    model=MODEL,
    tools=[fetch_article],
    system_prompt=(
        "You are a research writer. When given articles to analyze:\n"
        "1. Fetch each article\n"
        "2. Write key notes to files (e.g., /notes/react-pattern.md)\n"
        "3. After reading all articles, review your notes\n"
        "4. Write a synthesis to /output/synthesis.md\n"
        "5. Present the synthesis to the user\n\n"
        "Use the filesystem to organize your work — don't try to hold everything in memory."
    ),
)

result = writer_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Fetch and analyze two articles: one on the ReAct pattern (url: react-pattern) "
        "and one on agent memory systems (url: agent-memory). "
        "Write notes on each, then synthesize the key themes across both."
    }]}
)

show(result)

In [ ]:
# See what files the agent created
show_steps(result)

**Why this matters:** The agent used files as **external working memory**. Instead of keeping two full articles in its context window, it:
1. Read article → wrote notes → freed the context
2. Read article → wrote notes → freed the context
3. Read back *just the notes* → synthesized

This is the same pattern Claude Code uses with its `Read`/`Write` tools, and the same pattern OpenClaw uses with its memory files. **Files are the agent's notebook.**

---

### ✏️ Exercise 2: Persistent Backend

The default `StateBackend` is ephemeral — files disappear when the session ends. Swap it to a `FilesystemBackend` that writes to a real directory on your Colab instance.

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir="/content/agent_workspace", virtual_mode=True)
agent = create_deep_agent(model=MODEL, tools=[...], backend=backend)
```

**Question to think about:** What are the security implications of giving an agent access to your real filesystem? When would you want this vs. the ephemeral default?

In [ ]:
# WRITE YOUR RESPONSE HERE

---
## Part 4: Subagents — Divide and Conquer

When a task has distinct subtasks, you can **delegate** to specialized subagents. Each subagent:
- Gets its own context window (isolation)
- Has its own tools and system prompt (specialization)
- Returns a compressed result to the parent (efficiency)

This is the **orchestrator-worker** pattern from lecture. The main agent plans and coordinates; subagents do the focused work.

In [ ]:
from deepagents import create_deep_agent


# ── Define tools for the subagents ────────────────────────────────

def analyze_code_quality(code: str) -> str:
    """Analyze a code snippet for quality issues. Returns a list of findings."""
    findings = []
    if len(code.split("\n")) > 50:
        findings.append("Function is too long (>50 lines). Consider breaking it up.")
    if "except:" in code or "except Exception:" in code:
        findings.append("Bare except clause found. Catch specific exceptions.")
    if code.count("if ") > 5:
        findings.append("High cyclomatic complexity. Consider refactoring nested conditionals.")
    if "# TODO" in code:
        findings.append("Unresolved TODO comments found.")
    if not code.strip().startswith(('def ', 'class ', 'import ', 'from ', '#')):
        findings.append("Code doesn't start with a standard Python construct.")
    if not findings:
        findings.append("Code looks clean! No major issues detected.")
    return "\n".join(f"- {f}" for f in findings)


def check_security(code: str) -> str:
    """Check code for common security issues. Returns security findings."""
    issues = []
    if "eval(" in code:
        issues.append("CRITICAL: eval() usage found — arbitrary code execution risk")
    if "os.system(" in code:
        issues.append("WARNING: os.system() found — use subprocess.run() instead")
    if "password" in code.lower() and "=" in code:
        issues.append("WARNING: Possible hardcoded password")
    if "pickle.load" in code:
        issues.append("WARNING: pickle.load on untrusted data is unsafe")
    if "input(" in code and "sql" in code.lower():
        issues.append("CRITICAL: Possible SQL injection — user input near SQL operations")
    if not issues:
        issues.append("No security issues detected.")
    return "\n".join(f"- {i}" for i in issues)


# ── Create an agent with subagents ─────────────────────────────────
review_agent = create_deep_agent(
    model=MODEL,
    tools=[],  # The main agent coordinates — subagents do the work
    system_prompt=(
        "You are a code review coordinator. When asked to review code:\n"
        "1. Use the 'quality_reviewer' subagent for code quality analysis\n"
        "2. Use the 'security_auditor' subagent for security review\n"
        "3. Synthesize both reports into a final review\n"
        "Always delegate to BOTH subagents before writing your synthesis."
    ),
    subagents=[
        {
            "name": "quality_reviewer",
            "description": "Reviews code for quality issues: style, complexity, maintainability",
            "tools": [analyze_code_quality],
            "system_prompt": "You are a code quality expert. Analyze the given code and provide detailed feedback on quality, readability, and maintainability.",
        },
        {
            "name": "security_auditor",
            "description": "Reviews code for security vulnerabilities and unsafe patterns",
            "tools": [check_security],
            "system_prompt": "You are a security expert. Analyze the given code for vulnerabilities, unsafe patterns, and security best practices.",
        },
    ],
)

# ── Give it code to review ─────────────────────────────────────────
sample_code = '''
import os
import pickle

def process_user_data(filename):
    # TODO: add input validation
    password = "admin123"
    data = pickle.load(open(filename, "rb"))
    if data:
        if data.get("role") == "admin":
            if data.get("action") == "delete":
                if data.get("confirmed"):
                    if data.get("target"):
                        if os.path.exists(data["target"]):
                            os.system(f"rm -rf {data['target']}")
    return "done"
'''

result = review_agent.invoke(
    {"messages": [{"role": "user", "content": f"Please review this code:\n```python\n{sample_code}\n```"}]}
)

show(result)

In [ ]:
# See the delegation pattern
show_steps(result)

**The pattern:**
- The **coordinator** doesn't analyze code itself — it delegates to specialists
- Each **subagent** has focused tools and expertise
- Results come back **compressed** — the coordinator only sees summaries, not the full analysis trace
- The coordinator **synthesizes** multiple perspectives into one coherent review

This is exactly how production systems work. Claude Code uses subagents for file search, code analysis, and test running. The orchestrator holds the big picture; workers handle details.

---

---
## Part 5: Putting It All Together — Build a Research Agent

Now you'll combine everything:
- **Tools** for data gathering
- **Planning** to organize multi-step work
- **Filesystem** to manage context
- **Subagents** for specialized tasks

### The Task

Build an agent that can research a topic and produce a structured report. The agent should:
1. Plan its research approach
2. Gather information using tools
3. Delegate analysis to a specialized subagent
4. Write notes and a final report to the filesystem
5. Present the report to the user

In [ ]:
from deepagents import create_deep_agent


# ── Research tools ─────────────────────────────────────────────────

if HAS_TAVILY:
    from tavily import TavilyClient
    tavily = TavilyClient()

    def web_search(query: str) -> str:
        """Search the web for information on a topic. Returns relevant snippets."""
        results = tavily.search(query, max_results=3)
        output = []
        for r in results.get("results", []):
            output.append(f"**{r['title']}**\n{r['url']}\n{r['content'][:300]}\n")
        return "\n---\n".join(output) if output else "No results found."
else:
    def web_search(query: str) -> str:
        """Search the web for information on a topic. Returns relevant snippets."""
        mock_results = {
            "agent": (
                "**AI Agents in 2026: State of the Art**\n"
                "Agent frameworks have converged on a common pattern: tool-calling loops with "
                "planning, memory, and multi-agent coordination. Key players include LangChain "
                "DeepAgent, OpenAI Agents SDK, Anthropic Claude Code, and open-source alternatives "
                "like AutoGen and CrewAI.\n\n"
                "**Key Trends:**\n"
                "- Filesystem-based context management replacing pure conversation memory\n"
                "- Subagent delegation for context isolation\n"
                "- Built-in planning tools (todo lists, task decomposition)\n"
                "- Human-in-the-loop for high-stakes decisions"
            ),
            "default": f"Mock search results for: {query}. Enable Tavily for real web search.",
        }
        for key, result in mock_results.items():
            if key in query.lower():
                return result
        return mock_results["default"]


def get_framework_details(name: str) -> str:
    """Get technical details about an AI agent framework by name."""
    frameworks = {
        "deepagent": {
            "name": "LangChain DeepAgent",
            "released": "March 2026",
            "language": "Python",
            "core_features": ["Planning (write_todos)", "Virtual filesystem", "Subagent spawning", "Pluggable backends"],
            "built_on": "LangGraph runtime",
            "model_support": ["Anthropic", "OpenAI", "Google", "Azure", "AWS Bedrock", "HuggingFace"],
            "best_for": "Building custom agents with full control over architecture",
        },
        "claude code": {
            "name": "Claude Code (Anthropic)",
            "released": "2025",
            "language": "TypeScript (CLI), Python/any (usage)",
            "core_features": ["File read/write/edit", "Bash execution", "Agent spawning", "Hook system", "Skills"],
            "built_on": "Anthropic API",
            "model_support": ["Claude models only"],
            "best_for": "Software development, code review, project management",
        },
        "openai agents": {
            "name": "OpenAI Agents SDK",
            "released": "2025",
            "language": "Python",
            "core_features": ["Handoff pattern", "Guardrails", "Tracing", "MCP support"],
            "built_on": "OpenAI API",
            "model_support": ["OpenAI models (GPT-4o, o1, etc.)"],
            "best_for": "Multi-agent workflows with safety guardrails",
        },
    }
    for key, info in frameworks.items():
        if key in name.lower():
            return json.dumps(info, indent=2)
    return f"Framework '{name}' not found. Available: {list(frameworks.keys())}"


# ── Build the full research agent ──────────────────────────────────
full_agent = create_deep_agent(
    model=MODEL,
    tools=[web_search, get_framework_details],
    system_prompt=(
        "You are a thorough research agent. For any research task:\n\n"
        "1. PLAN: Use write_todos to outline your research steps\n"
        "2. GATHER: Use your tools to collect information\n"
        "3. ORGANIZE: Write notes to files as you go (e.g., /notes/topic.md)\n"
        "4. DELEGATE: Use the 'analyst' subagent for comparative analysis\n"
        "5. SYNTHESIZE: Write a final report to /report/final.md\n"
        "6. PRESENT: Share the key findings with the user\n\n"
        "Always write to files rather than holding everything in context."
    ),
    subagents=[
        {
            "name": "analyst",
            "description": "Comparative analysis specialist — give it data and it finds patterns, differences, and recommendations",
            "tools": [],
            "system_prompt": (
                "You are an expert analyst. When given research data, produce:\n"
                "1. A comparison table of key dimensions\n"
                "2. Strengths and weaknesses of each item\n"
                "3. A clear recommendation with reasoning"
            ),
        }
    ],
)


# ── Run the full pipeline ──────────────────────────────────────────
result = full_agent.invoke(
    {"messages": [{"role": "user", "content":
        "Research and compare three AI agent frameworks: LangChain DeepAgent, "
        "Claude Code, and OpenAI Agents SDK. I need to understand which would be "
        "best for a graduate-level course project. Consider: ease of learning, "
        "flexibility, documentation quality, and Python support."
    }]}
)

show(result)

In [ ]:
# Full trace — see planning, tool calls, file writes, subagent delegation
show_steps(result)

---
## Recap: The Four Capabilities

| Capability | What It Does | DeepAgent Feature | Why It Matters |
|---|---|---|---|
| **Tools** | Connect the agent to the world | `tools=[...]` parameter | Without tools, an agent can only generate text |
| **Planning** | Break complex tasks into steps | `write_todos` (built-in) | Prevents the agent from getting lost in multi-step tasks |
| **Filesystem** | External working memory | `write_file`, `read_file`, etc. | Overcomes context window limits |
| **Subagents** | Delegate specialized work | `subagents=[...]` parameter | Context isolation + specialization |

Every production agent framework has its own version of these four capabilities. The names change; the patterns don't.

---

## 🚀 Challenge: Build Your Own Agent

Pick one of these project ideas (or propose your own) and build an agent that uses **at least 3 of the 4 capabilities** above:

### Option A: Study Group Agent
An agent that helps students study for exams. It should:
- Accept a topic and generate practice questions (tool)
- Track which topics have been covered (planning)
- Save a study log to files (filesystem)

### Option B: Code Tutor Agent  
An agent that teaches programming concepts. It should:
- Generate code examples and explanations (tools)
- Delegate code review to a subagent (subagents)
- Maintain a lesson plan (planning)
- Save student progress notes (filesystem)

### Option C: Literature Review Agent
An agent that surveys research on a topic. It should:
- Search for papers (tools, optionally with Tavily)
- Plan a systematic review process (planning)
- Delegate summarization to a subagent (subagents)
- Write structured notes and a final synthesis (filesystem)

### Evaluation Criteria
- Does the agent use tools effectively?
- Does it plan before executing?
- Does it manage context (files or subagents)?
- Is the system prompt clear and specific?
- Does it produce useful output?

In [ ]:
# YOUR AGENT HERE
#
# Starter template:
#
# from deepagents import create_deep_agent
#
# def my_tool_1(arg: str) -> str:
#     """Description of what this tool does."""
#     ...
#
# def my_tool_2(arg: str) -> str:
#     """Description of what this tool does."""
#     ...
#
# agent = create_deep_agent(
#     model=MODEL,
#     tools=[my_tool_1, my_tool_2],
#     system_prompt="...",
#     subagents=[{"name": "...", "description": "...", "tools": [...], "system_prompt": "..."}],
# )
#
# result = agent.invoke({"messages": [{"role": "user", "content": "..."}]})
# show(result)

---
## Appendix: Key Concepts Reference

| Term | Definition |
|---|---|
| **Agent loop** | The cycle of: receive input → reason → call tool → observe result → repeat |
| **Tool** | A function the LLM can call to interact with the world |
| **System prompt** | Instructions that define the agent's behavior and personality |
| **Context window** | The total amount of text the LLM can "see" at once |
| **Planning** | Breaking a complex task into manageable steps before executing |
| **Subagent** | A child agent spawned for a specific subtask, with its own context |
| **Backend** | Where the agent's virtual filesystem stores files (memory, disk, cloud) |
| **Middleware** | Components that process messages between the user and the agent |
| **Human-in-the-loop** | Pausing agent execution for human approval on critical actions |
| **ReAct** | Reasoning + Acting — the dominant pattern for tool-using agents |

### Further Reading
- [DeepAgent Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [LangGraph Runtime](https://docs.langchain.com/oss/python/langgraph/overview)
- [DeepAgent Examples](https://github.com/langchain-ai/deepagents/tree/main/examples)